<a href="https://colab.research.google.com/github/jamesemcnally/critical-listener/blob/main/average_embeddings_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Mount Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- GPU check (not strictly needed without the model, but harmless to confirm) ---
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# --- Imports ---
import pandas as pd
import numpy as np

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# --- Load masked cleaned dataset ---
df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")
print(f"Loaded: {len(df):,} reviews")
print(df.columns.tolist())

# --- Load masked Nomic embeddings (with prefix) ---
embeddings_masked = np.load(f"{DRIVE_DIR}/nomic_masked_with_prefix.npy")
print(f"Embeddings: {embeddings_masked.shape}")
assert embeddings_masked.shape[0] == len(df), "Embeddings/df row mismatch — check alignment"

# --- Pitchfork-only mask + index, reused across recommenders ---
pitchfork_mask = df["dataset"] == "pitchfork"
pitchfork_idx = df[pitchfork_mask].index.tolist()
print(f"Pitchfork reviews: {len(pitchfork_idx):,}")

Mounted at /content/drive
GPU available: True
Device: NVIDIA L4
Loaded: 48,189 reviews
['dataset', 'review_id', 'entity_id', 'text', 'rating', 'album', 'artist', 'reviewer_name', 'reviewer_id', 'reviewer_type', 'reviewer_karma', 'published_on', 'source', 'source_url', 'cleaned_text', 'word_count', 'char_count', 'sentence_count', 'sentiment_score', 'sentiment_category', 'genre', 'ra_recommends', 'artist_norm', 'album_norm', 'input_with_prefix', 'input_no_prefix', 'personnel', 'cleaned_text_masked']
Embeddings: (48189, 768)
Pitchfork reviews: 22,810


In [ ]:
# --- Review count distribution per album ---
review_counts = df.groupby(["artist_norm", "album_norm"]).size().rename("n_reviews")

print(review_counts.value_counts().sort_index())
print(f"\nTotal unique albums: {review_counts.shape[0]:,}")
print(f"Albums with >1 review: {(review_counts > 1).sum():,} ({(review_counts > 1).mean():.1%})")

# Multi-source check — the more interesting case for the register confound
n_sources = df.groupby(["artist_norm", "album_norm"])["dataset"].nunique()
print(f"Albums with >1 source: {(n_sources > 1).sum():,} ({(n_sources > 1).mean():.1%})")

print("\nTop 10 albums by review count:")
print(review_counts.sort_values(ascending=False).head(10))

n_reviews
1    41423
2     3113
3      180
Name: count, dtype: int64

Total unique albums: 44,716
Albums with >1 review: 3,293 (7.4%)
Albums with >1 source: 3,293 (7.4%)

Top 10 albums by review count:
artist_norm            album_norm             
the chemical brothers  we are the night           3
radian                 chimeric                   3
jimmy edgar            majenta                    3
atoms for peace        amok                       3
paul white             rapping with paul white    3
hot chip               in our heads               3
                       made in the dark           3
                       one life stand             3
dj shadow              the outsider               3
scuba                  personality                3
Name: n_reviews, dtype: int64


In [ ]:
# --- Distribution of PF-only vs. cross-source-averaged embedding similarity ---
# For every multi-review album, compares the Pitchfork-only embedding to the
# averaged-and-renormalized cross-source embedding. Tells you, across the
# full 3,293 albums, how much aggregation shifts the embedding from what a
# PF-only query pool would have given you.

multi_review_keys = review_counts[review_counts > 1].index

records = []
for artist_norm, album_norm in multi_review_keys:
    rows = df[(df["artist_norm"] == artist_norm) & (df["album_norm"] == album_norm)]
    pf_row = rows[rows["dataset"] == "pitchfork"]

    if pf_row.empty:
        continue  # skip if this multi-review album has no PF review to compare against

    pf_vec = embeddings_masked[pf_row.index[0]]
    avg_vec = embeddings_masked[rows.index].mean(axis=0)
    avg_vec = avg_vec / np.linalg.norm(avg_vec)
    cos_sim = float(pf_vec @ avg_vec)

    records.append({
        "artist_norm": artist_norm,
        "album_norm": album_norm,
        "n_reviews": len(rows),
        "sources": sorted(rows["dataset"].unique().tolist()),
        "cos_sim_pf_vs_avg": cos_sim
    })

shift_df = pd.DataFrame(records)
print(f"Albums compared: {len(shift_df):,} (of {len(multi_review_keys):,} multi-review albums)")

print("\nCosine similarity (PF-only vs. cross-source average) — summary:")
print(shift_df["cos_sim_pf_vs_avg"].describe())

print("\nAlbums with similarity < 0.95 (notable shift):")
low_sim = shift_df[shift_df["cos_sim_pf_vs_avg"] < 0.95].sort_values("cos_sim_pf_vs_avg")
print(f"Count: {len(low_sim):,} ({len(low_sim)/len(shift_df):.1%})")
print(low_sim.head(10))

# Breakdown by source combination — does RA-involvement specifically drive bigger shifts?
shift_df["source_combo"] = shift_df["sources"].apply(lambda s: ", ".join(s))
print("\nMean similarity by source combination:")
print(shift_df.groupby("source_combo")["cos_sim_pf_vs_avg"].agg(["mean", "median", "count"]))

Albums compared: 3,214 (of 3,293 multi-review albums)

Cosine similarity (PF-only vs. cross-source average) — summary:
count    3214.000000
mean        0.978986
std         0.008937
min         0.899190
25%         0.975689
50%         0.980570
75%         0.984639
max         0.993697
Name: cos_sim_pf_vs_avg, dtype: float64

Albums with similarity < 0.95 (notable shift):
Count: 41 (1.3%)
                   artist_norm                  album_norm  n_reviews  \
1250             joanna newsom              have one on me          2   
831                erykah badu                  mama’s gun          2   
1887                   nirvana                 incesticide          2   
612                 de la soul      3 feet high and rising          2   
1688                   maxwell  maxwell’s urban hang suite          2   
2163     red hot chili peppers       blood sugar sex magik          2   
235        belle and sebastian                   tigermilk          2   
382          bruce sprin

In [ ]:
# --- Inspect the lowest-similarity case: Joanna Newsom — Have One on Me ---

case = df[
    (df["artist_norm"] == "joanna newsom") &
    (df["album_norm"] == "have one on me")
]

for _, row in case.iterrows():
    print(f"\n{'='*70}")
    print(f"Source: {row['dataset']}  |  Rating: {row['rating']}  |  Reviewer: {row['reviewer_name']}")
    print(f"{'='*70}")
    print(row["cleaned_text"][:1500])


Source: critique_brainz  |  Rating: 5.0  |  Reviewer: gleech
TL;DR: Its grand theme is the closeness of love to many terrible emotions: fear, possession, weakness, unfreedom, subsumption, vengeance, amoral avarice. The key symbol is _love as intoxication as poisoning_; heartbreak as the inevitable hangover, passion as compulsive vice, love as gluttony. "Have one on me" is disguised aggression towards a neglectful/abusive lover: "drink this gall, traitor". (See also "_go long, right over the edge of the earth_".) Give love a little shove and it becomes terror _Have One On Me_ is a map of the heart, and you shouldn't expect those to offer themselves lightly. It exhausts all the forms of love: _divine_ or _agape_ (tracks 3, 7, 14); _filial_ (track 9, 14); _courtly_ (track 2); _obsessional_ (tracks 1, 5, 10); _maternal_ (track 6! but touches in 1, 5, 11, 14); _platonic_ (passionate friendship: track 8 and maybe 11); _panicked_ (track 4); _dependent_ (track 5, 10, 16); _wilful_ (track 1, 1

In [ ]:
# --- Inspect additional low-similarity CB+PF pairs ---

pairs_to_check = [
    ("erykah badu", "mama’s gun"),
    ("nirvana", "incesticide"),
    ("de la soul", "3 feet high and rising"),
]

for artist_norm, album_norm in pairs_to_check:
    case = df[(df["artist_norm"] == artist_norm) & (df["album_norm"] == album_norm)]
    for _, row in case.iterrows():
        print(f"\n{'='*70}")
        print(f"{artist_norm} — {album_norm}")
        print(f"Source: {row['dataset']}  |  Rating: {row['rating']}  |  Reviewer: {row['reviewer_name']}")
        print(f"{'='*70}")
        print(row["cleaned_text"][:1200])


erykah badu — mama’s gun
Source: critique_brainz  |  Rating: nan  |  Reviewer: Daryl Easlea
When Erykah Badu's debut album, Baduizm, was released in 1997, with her headdress, Billie Holiday vocal inflections and her smart, spot-on reinterpretation of R&B, she became the high-profile female figurehead for neo-soul. When Mama's Gun arrived three years later, it carried huge levels of expectation. Follow-up albums are forever fascinating. Often, you are damned if you do and damned if you don't. If Badu had given the world another Baduizm, everyone would have accused her of simply standing still. Instead, she flew freely and deeply into alternative arenas and released a meandering, occasionally impenetrable masterpiece. Produced by the Soulquarians collective, led by drummer Ahmir "Questlove" Thompson, rappers Common, Bilal et al, Mama's Gun is both a step away and continuation of the hippy Afrocentricity of Baduizm. At times, it resembles an out-of-focus Sly Stone jam; at others the best

In [ ]:
def build_album_embeddings(df, embeddings, key_cols=("artist_norm", "album_norm"), length_flag_threshold=0.30):
    groups = df.groupby(list(key_cols)).indices

    album_keys, n_reviews, sources, length_flags = [], [], [], []
    album_embeddings = np.zeros((len(groups), embeddings.shape[1]), dtype=embeddings.dtype)

    for i, (key, row_positions) in enumerate(groups.items()):
        vecs = embeddings[row_positions]
        mean_vec = vecs.mean(axis=0)
        album_embeddings[i] = mean_vec / np.linalg.norm(mean_vec)
        album_keys.append(key)

        rows = df.iloc[row_positions]
        n_reviews.append(len(rows))
        sources.append(sorted(rows["dataset"].unique().tolist()))

        # New: length flag, only meaningful when there's a PF review + at least one other
        pf_wc = rows.loc[rows["dataset"] == "pitchfork", "word_count"]
        other_wc = rows.loc[rows["dataset"] != "pitchfork", "word_count"]
        if len(rows) > 1 and not pf_wc.empty and not other_wc.empty:
            ratio = other_wc.min() / pf_wc.iloc[0]
            length_flags.append(ratio < length_flag_threshold)
        else:
            length_flags.append(False)

    album_df = pd.DataFrame(album_keys, columns=list(key_cols))
    album_df["n_reviews"] = n_reviews
    album_df["sources"] = sources
    album_df["length_flag"] = length_flags

    return album_df, album_embeddings

In [ ]:
def recommend_from_album_embeddings(query_artist_norm, query_album_norm, album_df, album_embeddings, k=5):
    query_mask = (album_df["artist_norm"] == query_artist_norm) & (album_df["album_norm"] == query_album_norm)
    if not query_mask.any():
        print(f"Not found: {query_artist_norm} — {query_album_norm}")
        return None

    query_idx = album_df[query_mask].index[0]
    query_vec = album_embeddings[query_idx]

    sims = album_embeddings @ query_vec
    sims[query_idx] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen_artists = {query_artist_norm}
    results = []
    for idx in ranked:
        artist = album_df.loc[idx, "artist_norm"]
        if artist in seen_artists:
            continue
        results.append({
            "artist": album_df.loc[idx, "artist_norm"],
            "album": album_df.loc[idx, "album_norm"],
            "score": round(float(sims[idx]), 4),
            "n_reviews": album_df.loc[idx, "n_reviews"],
            "length_flag": album_df.loc[idx, "length_flag"]
        })
        seen_artists.add(artist)
        if len(results) >= k:
            break

    return results

### Testing the model

In [ ]:
# --- Test: does review length explain the PF-vs-avg similarity drop? ---
# For each multi-review album, compute the word-count gap between the PF review
# and the other review(s), and check correlation with cos_sim_pf_vs_avg.

length_records = []
for artist_norm, album_norm in multi_review_keys:
    rows = df[(df["artist_norm"] == artist_norm) & (df["album_norm"] == album_norm)]
    pf_row = rows[rows["dataset"] == "pitchfork"]
    other_rows = rows[rows["dataset"] != "pitchfork"]

    if pf_row.empty or other_rows.empty:
        continue

    pf_wc = pf_row.iloc[0]["word_count"]
    other_wc_mean = other_rows["word_count"].mean()

    length_records.append({
        "artist_norm": artist_norm,
        "album_norm": album_norm,
        "pf_word_count": pf_wc,
        "other_word_count_mean": other_wc_mean,
        "word_count_ratio": other_wc_mean / pf_wc,   # <1 means non-PF review(s) shorter
        "word_count_diff": pf_wc - other_wc_mean      # positive means PF longer
    })

length_df = pd.DataFrame(length_records)

# Merge with the similarity results from before
merged = shift_df.merge(length_df, on=["artist_norm", "album_norm"])

print(f"Merged rows: {len(merged):,}")

# Correlation: does a bigger length gap predict lower similarity?
corr_diff = merged["word_count_diff"].corr(merged["cos_sim_pf_vs_avg"])
corr_ratio = merged["word_count_ratio"].corr(merged["cos_sim_pf_vs_avg"])
print(f"\nCorrelation (PF longer than other → lower similarity): {corr_diff:.3f}")
print(f"Correlation (word_count_ratio vs. similarity):          {corr_ratio:.3f}")

# Compare length stats: low-similarity tail vs. the rest
low_tail = merged[merged["cos_sim_pf_vs_avg"] < 0.95]
rest = merged[merged["cos_sim_pf_vs_avg"] >= 0.95]

print(f"\nLow-similarity tail (n={len(low_tail)}):")
print(f"  Mean PF word count:    {low_tail['pf_word_count'].mean():.0f}")
print(f"  Mean other word count: {low_tail['other_word_count_mean'].mean():.0f}")
print(f"  Mean ratio (other/PF): {low_tail['word_count_ratio'].mean():.2f}")

print(f"\nRest (n={len(rest)}):")
print(f"  Mean PF word count:    {rest['pf_word_count'].mean():.0f}")
print(f"  Mean other word count: {rest['other_word_count_mean'].mean():.0f}")
print(f"  Mean ratio (other/PF): {rest['word_count_ratio'].mean():.2f}")

Merged rows: 3,214

Correlation (PF longer than other → lower similarity): -0.437
Correlation (word_count_ratio vs. similarity):          0.252

Low-similarity tail (n=41):
  Mean PF word count:    1889
  Mean other word count: 432
  Mean ratio (other/PF): 0.31

Rest (n=3173):
  Mean PF word count:    686
  Mean other word count: 382
  Mean ratio (other/PF): 0.61


In [ ]:
# --- Find the word-count-ratio threshold that best predicts low-similarity tail ---
# Tests a range of cutoffs on word_count_ratio (other/PF) and word_count_diff,
# scoring each by how well it separates the low-sim tail (<0.95) from the rest.

from sklearn.metrics import precision_recall_curve, roc_auc_score

merged["is_low_sim"] = (merged["cos_sim_pf_vs_avg"] < 0.95).astype(int)

print(f"Low-sim cases: {merged['is_low_sim'].sum()} / {len(merged)} ({merged['is_low_sim'].mean():.1%})\n")

# AUC for each candidate predictor (lower ratio / higher diff = more "PF dominates" = more risk)
# Flip ratio sign convention so "higher score = higher risk" for both, to compare fairly
auc_ratio = roc_auc_score(merged["is_low_sim"], -merged["word_count_ratio"])
auc_diff  = roc_auc_score(merged["is_low_sim"], merged["word_count_diff"])
auc_pf_wc = roc_auc_score(merged["is_low_sim"], merged["pf_word_count"])

print(f"AUC using word_count_ratio (inverted): {auc_ratio:.3f}")
print(f"AUC using word_count_diff:             {auc_diff:.3f}")
print(f"AUC using pf_word_count alone:         {auc_pf_wc:.3f}")

# Scan threshold values on word_count_ratio to find best precision/recall tradeoff
print("\nThreshold scan on word_count_ratio (flag if ratio < threshold):")
print(f"{'threshold':>10} {'flagged':>8} {'precision':>10} {'recall':>8}")
for thresh in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    flagged = merged["word_count_ratio"] < thresh
    n_flagged = flagged.sum()
    if n_flagged == 0:
        continue
    precision = merged.loc[flagged, "is_low_sim"].mean()
    recall = merged.loc[flagged, "is_low_sim"].sum() / merged["is_low_sim"].sum()
    print(f"{thresh:>10.2f} {n_flagged:>8} {precision:>10.1%} {recall:>8.1%}")

Low-sim cases: 41 / 3214 (1.3%)

AUC using word_count_ratio (inverted): 0.911
AUC using word_count_diff:             0.894
AUC using pf_word_count alone:         0.857

Threshold scan on word_count_ratio (flag if ratio < threshold):
 threshold  flagged  precision   recall
      0.20       92      32.6%    73.2%
      0.25      161      20.5%    80.5%
      0.30      230      14.8%    82.9%
      0.35      352       9.7%    82.9%
      0.40      543       6.4%    85.4%
      0.45      805       4.3%    85.4%
      0.50     1122       3.2%    87.8%
